# Notebook 07 — Capacidade Instalada e Fator de Capacidade

**MVP de Engenharia de Dados** · PUC-Rio · Sprint 3
*Extensão do escopo original*

---

### Por que este notebook existe

A primeira versão deste trabalho registrou, como limitação, a ausência de dados de capacidade instalada — o que impedia calcular o **fator de capacidade** e, portanto, distinguir "gerou mais porque instalou mais" de "gerou mais porque aproveitou melhor".

Este notebook incorpora um segundo dataset do ONS para resolver essa limitação.

### Fonte

| | |
|---|---|
| **Dataset** | Capacidade Instalada de Geração |
| **Portal** | https://dados.ons.org.br/dataset/capacidade-geracao |
| **Licença** | Creative Commons Attribution (CC-BY) |
| **Granularidade** | Unidade geradora (mais fina que usina) |
| **Volume** | 5.686 unidades |

### O obstáculo e a solução

O dataset **não possui série histórica**: é um retrato da situação atual, atualizado diariamente. À primeira vista, isso inviabilizaria a análise de sete anos.

Mas o cadastro traz `dat_entradaoperacao` e `dat_desativacao` de cada unidade. Com esses dois campos é possível **reconstruir a capacidade instalada em qualquer data passada**: basta somar as unidades que já haviam entrado em operação e ainda não tinham sido desativadas naquele momento.

O retrato de hoje carrega a linha do tempo dentro dele.

## 1. Coleta e inspeção

Mesmo procedimento do notebook 01: download por código a partir da URL oficial, e inspeção das primeiras linhas **antes** de qualquer processamento.

In [0]:
import os, requests

VOLUME_BRONZE = "/Volumes/workspace/bronze/raw_ons"
URL_CAP = "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/capacidade-geracao/CAPACIDADE_GERACAO.csv"
destino = f"{VOLUME_BRONZE}/CAPACIDADE_GERACAO.csv"

if os.path.exists(destino):
    print(f"[JA EXISTE] {os.path.getsize(destino)/1024/1024:.2f} MB")
else:
    print("[BAIXANDO ]")
    r = requests.get(URL_CAP, timeout=600)
    r.raise_for_status()
    with open(destino, "wb") as f:
        f.write(r.content)
    print(f"[OK       ] {len(r.content)/1024/1024:.2f} MB")

# Mesma disciplina do notebook 01: olhar antes de processar.
print()
with open(destino, "r", encoding="utf-8") as f:
    for i in range(3):
        print(f"Linha {i}: {f.readline().rstrip()[:250]}")

[JA EXISTE] 1.22 MB

Linha 0: id_subsistema;nom_subsistema;id_estado;nom_estado;nom_modalidadeoperacao;nom_agenteproprietario;nom_agenteoperador;nom_tipousina;nom_usina;ceg;nom_unidadegeradora;cod_equipamento;num_unidadegeradora;nom_combustivel;dat_entradateste;dat_entradaoperaca
Linha 1: NE;NORDESTE       ;AL;ALAGOAS;TIPO I;AXIA NORDESTE;AXIA NORDESTE;HIDROELÉTRICA;XINGÓ;UHE.PH.SE.027053-9.01;UG  527 MW USINA XINGO               1 AL;ALUXG-0UG1          ;1     ;HIDRÁULICA;1997-08-22;1997-08-22;;527.0
Linha 2: NE;NORDESTE       ;AL;ALAGOAS;TIPO I;AXIA NORDESTE;AXIA NORDESTE;HIDROELÉTRICA;XINGÓ;UHE.PH.SE.027053-9.01;UG  527 MW USINA XINGO               6 AL;ALUXG-0UG6          ;6     ;HIDRÁULICA;1994-04-30;1994-04-30;;527.0


A inspeção revelou dois pontos que exigem atenção.

**Separador `;`**, consistente com o outro dataset do ONS.

**Padding de largura fixa.** O campo `nom_subsistema` vem como `"NORDESTE            "`, preenchido até 20 caracteres. É um arquivo de largura fixa disfarçado de CSV. Sem `trim`, qualquer junção por subsistema falharia **silenciosamente**, porque `"NORDESTE"` e `"NORDESTE            "` são valores diferentes para o Spark.

A célula a seguir numera cada campo e delimita os valores com colchetes, tornando o padding visível.

In [0]:
with open(destino, "r", encoding="utf-8") as f:
    cabecalho = f.readline().rstrip().split(";")
    linha1    = f.readline().rstrip().split(";")

print(f"Total de colunas no cabeçalho: {len(cabecalho)}")
print(f"Total de campos na linha 1   : {len(linha1)}")
print()
for i, (col, val) in enumerate(zip(cabecalho, linha1)):
    print(f"{i:2d}  {col:28s} = [{val}]")

Total de colunas no cabeçalho: 18
Total de campos na linha 1   : 18

 0  id_subsistema                = [NE]
 1  nom_subsistema               = [NORDESTE       ]
 2  id_estado                    = [AL]
 3  nom_estado                   = [ALAGOAS]
 4  nom_modalidadeoperacao       = [TIPO I]
 5  nom_agenteproprietario       = [AXIA NORDESTE]
 6  nom_agenteoperador           = [AXIA NORDESTE]
 7  nom_tipousina                = [HIDROELÉTRICA]
 8  nom_usina                    = [XINGÓ]
 9  ceg                          = [UHE.PH.SE.027053-9.01]
10  nom_unidadegeradora          = [UG  527 MW USINA XINGO               1 AL]
11  cod_equipamento              = [ALUXG-0UG1          ]
12  num_unidadegeradora          = [1     ]
13  nom_combustivel              = [HIDRÁULICA]
14  dat_entradateste             = [1997-08-22]
15  dat_entradaoperacao          = [1997-08-22]
16  dat_desativacao              = []
17  val_potenciaefetiva          = [527.0]


**Segunda discrepância entre documentação e arquivo neste projeto.** O dicionário oficial lista 19 campos; o CSV tem 18. O campo `id_ons` está documentado mas não existe no arquivo.

Somada ao separador divergente encontrado no notebook 01, forma-se um padrão: a documentação do portal descreve a intenção, o arquivo descreve a realidade. Validar contra o arquivo, e não contra o manual, deixou de ser precaução e virou método neste trabalho.

## 2. Camada Bronze

Mesmo princípio das outras camadas Bronze: todas as colunas como texto, apenas com metadados de linhagem acrescentados. A tipagem fica para a Silver.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

spark.conf.set("spark.sql.session.timeZone", "UTC")

COLUNAS = [
    "id_subsistema", "nom_subsistema", "id_estado", "nom_estado",
    "nom_modalidadeoperacao", "nom_agenteproprietario", "nom_agenteoperador",
    "nom_tipousina", "nom_usina", "ceg", "nom_unidadegeradora",
    "cod_equipamento", "num_unidadegeradora", "nom_combustivel",
    "dat_entradateste", "dat_entradaoperacao", "dat_desativacao",
    "val_potenciaefetiva",
]

# Bronze: tudo como texto, igual ao notebook 02.
schema_cap = StructType([StructField(c, StringType(), True) for c in COLUNAS])

df_cap_bronze = (
    spark.read
        .option("header", True).option("sep", ";").option("encoding", "UTF-8")
        .schema(schema_cap)
        .csv(f"{VOLUME_BRONZE}/CAPACIDADE_GERACAO.csv")
        .withColumn("_arquivo_origem", F.col("_metadata.file_name"))
        .withColumn("_data_ingestao", F.current_timestamp())
)

(df_cap_bronze.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.bronze.capacidade_geracao_bruto"))

print(f"Unidades geradoras: {df_cap_bronze.count():,}")

Unidades geradoras: 5,686


## 3. Descoberta das categorias

Antes de mapear os tipos de usina para as quatro fontes do Balanço de Energia, é preciso saber quais categorias realmente existem no arquivo — e não quais o dicionário diz existir.

In [0]:
%sql
SELECT
  TRIM(nom_tipousina)   AS tipo_usina,
  COUNT(*)              AS qtd_unidades,
  ROUND(SUM(CAST(val_potenciaefetiva AS DOUBLE)) / 1000, 1) AS potencia_total_gw,
  SUM(CASE WHEN TRIM(dat_desativacao) = '' OR dat_desativacao IS NULL
           THEN 1 ELSE 0 END) AS unidades_ativas
FROM workspace.bronze.capacidade_geracao_bruto
GROUP BY TRIM(nom_tipousina)
ORDER BY potencia_total_gw DESC;

tipo_usina,qtd_unidades,potencia_total_gw,unidades_ativas
HIDROELÉTRICA,819,109.7,819
TÉRMICA,1469,39.5,489
EOLIELÉTRICA,2157,33.8,2157
FOTOVOLTAICA,1239,22.4,1236
NUCLEAR,2,2.0,2


**Resultado:**

| Tipo | Unidades | Potência (GW) | Ativas |
|---|---|---|---|
| Hidroelétrica | 819 | 109,7 | 819 |
| Térmica | 1.469 | 39,5 | **489** |
| Eolielétrica | 2.157 | 33,8 | 2.157 |
| Fotovoltaica | 1.239 | 22,4 | 1.236 |
| Nuclear | 2 | 2,0 | 2 |

Dois pontos decisivos.

**Dois terços das unidades térmicas estão desativadas** (980 de 1.469). Isso confirma que o ONS **preserva** as unidades históricas no cadastro com a data de desativação preenchida — exatamente o que viabiliza a reconstrução. Sem isso, a capacidade térmica de 2019 apareceria idêntica à de hoje.

**`NUCLEAR` é categoria própria**, com 2 unidades e 2 GW (Angra 1 e 2). O Balanço de Energia tem apenas quatro colunas de geração e contabiliza a nuclear dentro da térmica. O mapeamento precisa refletir isso explicitamente.

## 4. Camada Silver

Três tratamentos:

**`trim` em todos os campos de texto**, para eliminar o padding de largura fixa.

**Tipagem** de datas e da potência efetiva.

**Mapeamento de tipo de usina para fonte**, alinhando o vocabulário deste dataset ao do Balanço de Energia:

| Tipo no cadastro | Fonte no Balanço |
|---|---|
| HIDROELÉTRICA | Hidráulica |
| TÉRMICA | Térmica |
| **NUCLEAR** | **Térmica** |
| EOLIELÉTRICA | Eólica |
| FOTOVOLTAICA | Fotovoltaica |

As três checagens antes da gravação existem para falhar cedo: se alguma retornar valor diferente de zero, o mapeamento ou o cast deixou casos para trás, e é melhor descobrir aqui do que na análise.

In [0]:
from pyspark.sql import functions as F

# Mapeamento de tipo de usina para as fontes do Balanço de Energia.
# NUCLEAR é agrupada em Térmica porque o Balanço do ONS tem apenas quatro
# colunas de geração e Angra aparece dentro da geração térmica.
MAPA_FONTE = {
    "HIDROELÉTRICA": "Hidráulica",
    "TÉRMICA":       "Térmica",
    "NUCLEAR":       "Térmica",
    "EOLIELÉTRICA":  "Eólica",
    "FOTOVOLTAICA":  "Fotovoltaica",
}

cap = spark.table("workspace.bronze.capacidade_geracao_bruto")

# O arquivo vem com padding de largura fixa: trim em tudo que é texto.
for c in ["id_subsistema", "nom_subsistema", "nom_tipousina", "nom_usina",
          "nom_combustivel", "dat_entradaoperacao", "dat_desativacao"]:
    cap = cap.withColumn(c, F.trim(F.col(c)))

expr_fonte = F.create_map([F.lit(x) for kv in MAPA_FONTE.items() for x in kv])

cap_silver = (
    cap
      .withColumn("nom_fonte",           expr_fonte[F.col("nom_tipousina")])
      .withColumn("dat_entradaoperacao", F.to_date("dat_entradaoperacao", "yyyy-MM-dd"))
      .withColumn("dat_desativacao",     F.to_date("dat_desativacao",     "yyyy-MM-dd"))
      .withColumn("val_potenciaefetiva", F.col("val_potenciaefetiva").cast("double"))
      .select("id_subsistema", "nom_subsistema", "nom_usina", "nom_tipousina", "nom_fonte",
              "dat_entradaoperacao", "dat_desativacao", "val_potenciaefetiva",
              "_arquivo_origem", "_data_ingestao")
)

# Checagens antes de gravar
print(f"Sem fonte mapeada        : {cap_silver.filter(F.col('nom_fonte').isNull()).count()}")
print(f"Sem data de operação     : {cap_silver.filter(F.col('dat_entradaoperacao').isNull()).count()}")
print(f"Potência nula ou negativa: {cap_silver.filter(F.col('val_potenciaefetiva').isNull() | (F.col('val_potenciaefetiva') <= 0)).count()}")

(cap_silver.write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("workspace.silver.capacidade_geracao"))

print(f"\nUnidades na Silver: {spark.table('workspace.silver.capacidade_geracao').count():,}")

Sem fonte mapeada        : 0
Sem data de operação     : 0
Potência nula ou negativa: 0

Unidades na Silver: 5,686


## 5. Reconstrução da série histórica

O `crossJoin` cruza os sete anos com as 5.686 unidades, gerando cada combinação ano-unidade. Os dois filtros então decidem se aquela unidade existia naquele ano:

- já havia entrado em operação (`dat_entradaoperacao <= data_ref`)
- ainda não tinha sido desativada (`dat_desativacao` nula ou posterior à data)

É o equivalente a rodar o relógio para trás usando as datas de evento — técnica que se aplica sempre que um cadastro atual registra quando cada item entrou e saiu.

**Data de referência: 30 de junho.** Usar 31 de dezembro superestimaria os anos de expansão acelerada, contando como disponível o ano inteiro uma usina que entrou em dezembro. O meio do ano aproxima melhor a capacidade média do período. É uma escolha, e como toda escolha metodológica, fica registrada.

In [0]:
from pyspark.sql import functions as F

# Um registro por ano do recorte
ANOS = spark.range(2019, 2026).withColumnRenamed("id", "ano")

cap = spark.table("workspace.silver.capacidade_geracao")

# Confere se os códigos de subsistema batem com os do outro dataset
print("Subsistemas no dataset de capacidade:")
cap.select("id_subsistema").distinct().orderBy("id_subsistema").show()

# A capacidade instalada em uma data é a soma das unidades que já haviam
# entrado em operação e ainda não tinham sido desativadas naquele momento.
# Usamos 30/06 como referência: aproxima a capacidade média do ano melhor
# que 31/12, que superestimaria em anos de expansão acelerada.
cap_hist = (
    ANOS.crossJoin(cap)
        .withColumn("data_ref", F.to_date(F.concat(F.col("ano").cast("string"), F.lit("-06-30"))))
        .filter(F.col("dat_entradaoperacao") <= F.col("data_ref"))
        .filter(F.col("dat_desativacao").isNull() | (F.col("dat_desativacao") > F.col("data_ref")))
        .groupBy("ano", "id_subsistema", "nom_fonte")
        .agg(F.sum("val_potenciaefetiva").alias("capacidade_mw"))
)

(cap_hist.write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("workspace.gold.fato_capacidade"))

print("\nCapacidade instalada nacional por ano (GW, sem a parcela paraguaia de Itaipu):")
display(
    spark.table("workspace.gold.fato_capacidade")
        .filter(F.col("id_subsistema") != "PY")
        .groupBy("ano").pivot("nom_fonte")
        .agg(F.round(F.sum("capacidade_mw") / 1000, 1))
        .orderBy("ano")
)

Subsistemas no dataset de capacidade:
+-------------+
|id_subsistema|
+-------------+
|            N|
|           NE|
|           PY|
|            S|
|           SE|
+-------------+


Capacidade instalada nacional por ano (GW, sem a parcela paraguaia de Itaipu):


ano,Eólica,Fotovoltaica,Hidráulica,Térmica
2019,14.1,1.8,99.5,27.1
2020,14.9,2.6,102.5,28.7
2021,17.6,3.0,102.5,28.4
2022,20.9,4.5,102.5,30.5
2023,25.0,8.8,102.6,31.8
2024,30.0,13.4,102.6,31.5
2025,32.6,16.7,102.6,34.4


**Resultado da reconstrução — capacidade instalada nacional (GW):**

| 2019 | 14,1 | 1,8 | 99,5 | 27,1 |
| 2022 | 20,9 | 4,5 | 102,5 | 30,5 |
| 2025 | 32,6 | 16,7 | 102,6 | 34,4 |

A eólica mais que dobrou, a solar cresceu nove vezes e a hidráulica cresceu apenas 3%, praticamente estagnada. É a mesma história que a análise de geração contou no notebook 06 — agora confirmada por um **dataset independente**, o que é uma validação cruzada entre fontes.

A consulta também revelou um quinto código de subsistema: **`PY`**, correspondente à metade paraguaia de Itaipu. Ele existe no cadastro de capacidade mas não no Balanço de Energia, que tem apenas NE, N, S e SE. Por isso foi excluído da tabela acima e do fator de capacidade: incluí-lo inflaria a capacidade hidráulica em cerca de 7 GW, uma parcela cuja geração não aparece no Balanço.

---

## 6. Fator de capacidade

O fator de capacidade é a razão entre a energia efetivamente gerada e a que seria gerada se o parque operasse continuamente na potência nominal. É o indicador que separa **expansão do parque** de **aproveitamento do parque**.

Como a geração está em MWmed (potência média) e a capacidade em MW, a divisão é direta.

O resultado é um **fator de capacidade anual aproximado**: usa a capacidade instalada em 30 de junho como proxy da capacidade média do ano. A aproximação é adequada para comparar fontes e anos, mas pode se afastar do valor exato em anos de expansão muito rápida.

**Faixas esperadas no Brasil**, para servir de referência crítica ao resultado:

| Fonte | Faixa típica |
|---|---|
| Hidráulica | 40 a 55% |
| Eólica | 35 a 45% (o Nordeste tem dos melhores ventos do mundo) |
| Térmica | 20 a 60%, conforme o despacho |
| Fotovoltaica | **20 a 30%** — limitado pelas horas de sol |

In [0]:
%sql
-- Fator de capacidade = geração média efetiva / capacidade instalada.
-- Exclui o subsistema PY (metade paraguaia de Itaipu), que existe no
-- cadastro de capacidade mas não no Balanço de Energia.
WITH ger AS (
  SELECT t.ano, f.nom_fonte,
         SUM(g.val_geracao_mwmed) / COUNT(DISTINCT t.sk_tempo) AS geracao_media_mw
  FROM workspace.gold.fato_geracao g
  JOIN workspace.gold.dim_tempo t ON t.sk_tempo = g.sk_tempo
  JOIN workspace.gold.dim_fonte f ON f.sk_fonte = g.sk_fonte
  GROUP BY 1, 2
),
cap AS (
  SELECT ano, nom_fonte, SUM(capacidade_mw) AS capacidade_mw
  FROM workspace.gold.fato_capacidade
  WHERE id_subsistema <> 'PY'
  GROUP BY 1, 2
)
SELECT g.ano, g.nom_fonte,
  ROUND(c.capacidade_mw / 1000, 1)                        AS capacidade_gw,
  ROUND(g.geracao_media_mw / 1000, 1)                     AS geracao_media_gw,
  ROUND(100.0 * g.geracao_media_mw / c.capacidade_mw, 1)  AS fator_capacidade_pct
FROM ger g
JOIN cap c ON c.ano = g.ano AND c.nom_fonte = g.nom_fonte
ORDER BY g.nom_fonte, g.ano;

ano,nom_fonte,capacidade_gw,geracao_media_gw,fator_capacidade_pct
2019,Eólica,14.1,6.1,43.2
2020,Eólica,14.9,6.2,41.4
2021,Eólica,17.6,8.3,46.9
2022,Eólica,20.9,9.3,44.4
2023,Eólica,25.0,10.9,43.6
2024,Eólica,30.0,12.2,40.8
2025,Eólica,32.6,13.2,40.4
2019,Fotovoltaica,1.8,0.5,28.1
2020,Fotovoltaica,2.6,0.6,22.6
2021,Fotovoltaica,3.0,0.8,28.7


### Resultado — e uma anomalia

Três das quatro fontes se comportam de forma exemplar:

| Fonte | Faixa observada | Leitura |
|---|---|---|
| Hidráulica | 41,5 a 49,2% | Dentro do esperado |
| Eólica | 40,4 a 46,9% | Alto e estável, coerente com o regime de ventos |
| Térmica | 25,9 a 57,1% | Variação típica de fonte de complemento |

E há uma confirmação independente muito forte: **2021 aparece nas duas pontas**. A hidráulica cai para 41,5%, o menor da série, enquanto a térmica salta para 57,1%, o maior. A crise hídrica reaparece aqui por um caminho totalmente distinto do usado no notebook 06 — lá por volume de energia, aqui por aproveitamento do parque instalado.

**A fotovoltaica, porém, quebra:**

| Ano | FC | Avaliação |
|---|---|---|
| 2019 | 28,1% | plausível |
| 2020 | 22,6% | plausível |
| 2021 | 28,7% | plausível |
| 2022 | 30,6% | no limite |
| **2023** | **55,7%** | **impossível** |
| 2024 | 62,4% | impossível |
| 2025 | 62,8% | impossível |

Um fator de capacidade de 62% em geração solar é fisicamente impossível: exigiria sol equivalente a 15 horas por dia, todos os dias do ano. Nem o deserto do Atacama chega perto.

E não se trata de um desvio constante, que sugeriria erro de cálculo. É uma **quebra estrutural entre 2022 e 2023** — o indicador quase dobra de um ano para o outro, e nenhuma tecnologia dobra de eficiência nesse prazo.

A consulta seguinte localiza o mês exato da quebra. Crescimento gradual indicaria expansão real; um degrau súbito indicaria mudança de escopo ou de metodologia.

In [0]:
%sql
-- Geração fotovoltaica mensal em 2022 e 2023.
-- Se houver mudança de metodologia, aparece como um degrau súbito
-- num mês específico, não como crescimento gradual.
SELECT t.ano, t.mes,
  ROUND(SUM(g.val_geracao_mwmed) / COUNT(DISTINCT t.sk_tempo), 0) AS geracao_media_mw
FROM workspace.gold.fato_geracao g
JOIN workspace.gold.dim_tempo t ON t.sk_tempo = g.sk_tempo
JOIN workspace.gold.dim_fonte f ON f.sk_fonte = g.sk_fonte
WHERE f.nom_fonte = 'Fotovoltaica' AND t.ano IN (2022, 2023)
GROUP BY t.ano, t.mes
ORDER BY t.ano, t.mes;

ano,mes,geracao_media_mw
2022,1,1100.0
2022,2,1190.0
2022,3,1193.0
2022,4,1208.0
2022,5,1130.0
2022,6,1151.0
2022,7,1283.0
2022,8,1479.0
2022,9,1679.0
2022,10,1798.0


### A quebra tem data

| Mês | Geração média (MW) |
|---|---|
| 2023-03 | 1.876 |
| 2023-04 | 2.084 |
| **2023-05** | **4.855** |
| 2023-06 | 4.737 |

Nos dezesseis meses anteriores a série subiu de 1.100 para 2.084 MW — crescimento gradual, compatível com usinas entrando em operação. Entre abril e maio de 2023, salta **2.771 MW em um único mês**.

Para que isso fosse expansão real, seria necessário instalar cerca de 9 GW de painéis em trinta dias. A capacidade solar despachada pelo ONS naquele ano inteiro era de 8,8 GW — ou seja, seria preciso **mais que dobrar todo o parque nacional em um mês**. Fisicamente impossível.

**Conclusão: há uma quebra estrutural a partir de maio de 2023, compatível com alteração de escopo ou de metodologia.**

A hipótese mais provável é a incorporação da micro e minigeração distribuída ao Balanço de Energia, sustentada pela própria descrição do dataset de capacidade, que cobre exclusivamente "unidades geradoras de usinas **despachadas pelo ONS**" — o que exclui geração distribuída. O numerador passou a incluir uma parcela que o denominador nunca conteve.

O que o dado **prova** é a mudança e a sua data. A causa específica permanece como hipótese fundamentada, não como fato estabelecido.

---

## Conclusão da extensão

**O que a extensão entregou.** A limitação registrada na versão original — ausência de dados de capacidade instalada — foi resolvida para três das quatro fontes. Hidráulica, eólica e térmica agora têm fator de capacidade calculado e interpretável ao longo de todo o recorte.

**O que a extensão descobriu.** Para a fotovoltaica, o indicador só é legível até 2022. A partir de maio de 2023 os dois datasets deixam de ser comparáveis, por diferença de escopo.

**O que isso implica para a análise anterior.** A Pergunta 1 do notebook 06 afirma que a geração fotovoltaica "multiplicou-se por 21" entre 2019 e 2025. O número reflete fielmente o que o ONS publica, mas **não pode ser lido como crescimento físico puro**: parte dele é expansão real e parte é ampliação do escopo contábil a partir de maio de 2023. A afirmação recebeu ressalva no notebook 06 e no README.

Uma extensão que apenas acrescenta uma métrica é útil. Uma que **encontra um limite na análise anterior** é mais valiosa, porque impede que um número correto seja interpretado de forma incorreta.

## Documentação no catálogo

In [0]:
catalogo_cap = {
 "workspace.bronze.capacidade_geracao_bruto": {
   "_tabela": "Camada Bronze da extensao. Cadastro de unidades geradoras despachadas pelo ONS, "
              "preservado como texto. Fonte: dados.ons.org.br, licenca CC-BY. 5.686 unidades.",
   "id_subsistema":          "Texto bruto. Codigo do subsistema da usina. Dominio: NE, N, S, SE, PY.",
   "nom_subsistema":         "Texto bruto. Nome do subsistema, com padding de largura fixa preservado.",
   "id_estado":              "Texto bruto. Sigla do estado onde a usina esta localizada.",
   "nom_estado":             "Texto bruto. Nome do estado por extenso.",
   "nom_modalidadeoperacao": "Texto bruto. Modalidade de operacao da usina. Dominio: TIPO I, TIPO II-A, TIPO II-B, TIPO II-C.",
   "nom_agenteproprietario": "Texto bruto. Agente proprietario da usina.",
   "nom_agenteoperador":     "Texto bruto. Agente operador da usina.",
   "nom_tipousina":          "Texto bruto. Tipo da usina. Dominio: HIDROELETRICA, TERMICA, EOLIELETRICA, FOTOVOLTAICA, NUCLEAR.",
   "nom_usina":              "Texto bruto. Nome da usina.",
   "ceg":                    "Texto bruto. Codigo Unico do Empreendimento de Geracao, estabelecido pela ANEEL.",
   "nom_unidadegeradora":    "Texto bruto. Nome da unidade geradora, com padding de largura fixa preservado.",
   "cod_equipamento":        "Texto bruto. Codigo do equipamento da unidade geradora.",
   "num_unidadegeradora":    "Texto bruto. Codigo operacional da unidade geradora.",
   "nom_combustivel":        "Texto bruto. Combustivel da unidade geradora.",
   "dat_entradateste":       "Texto bruto. Data de liberacao para entrada em comissionamento, formato aaaa-MM-dd.",
   "dat_entradaoperacao":    "Texto bruto. Data de liberacao para operacao comercial. Convertida para date na Silver e usada na reconstrucao da serie historica.",
   "dat_desativacao":        "Texto bruto. Data de desativacao. Vem vazia quando a unidade esta ativa. Usada na reconstrucao da serie historica.",
   "val_potenciaefetiva":    "Texto bruto. Potencia nominal da unidade geradora em MW, conforme documento normativo da ANEEL.",
   "_arquivo_origem":        "Metadado de linhagem. Nome do arquivo CSV que originou o registro.",
   "_data_ingestao":         "Metadado de linhagem. Momento da carga na camada Bronze.",
 },
 "workspace.silver.capacidade_geracao": {
   "_tabela": "Camada Silver da extensao. Cadastro tipado, com padding de largura fixa removido "
              "e tipo de usina mapeado para as fontes do Balanco de Energia.",
   "id_subsistema":       "Codigo do subsistema. Dominio: NE, N, S, SE, PY. O codigo PY corresponde a metade paraguaia de Itaipu e nao existe no Balanco de Energia.",
   "nom_subsistema":      "Nome do subsistema por extenso.",
   "nom_usina":           "Nome da usina.",
   "nom_tipousina":       "Tipo original no cadastro. Dominio: HIDROELETRICA, TERMICA, EOLIELETRICA, FOTOVOLTAICA, NUCLEAR.",
   "nom_fonte":           "Tipo mapeado para o vocabulario do Balanco de Energia. NUCLEAR e agrupada em Termica, coerente com a contabilizacao do ONS.",
   "dat_entradaoperacao": "Data de liberacao para operacao comercial. Usada na reconstrucao da serie historica.",
   "dat_desativacao":     "Data de desativacao. Nulo indica unidade ainda ativa. Usada na reconstrucao da serie historica.",
   "val_potenciaefetiva": "Potencia nominal da unidade geradora, em MW. Dominio: maior que zero.",
   "_arquivo_origem":     "Metadado de linhagem. Nome do arquivo CSV de origem.",
   "_data_ingestao":      "Metadado de linhagem. Momento da carga na camada Bronze.",
 },
 "workspace.gold.fato_capacidade": {
   "_tabela": "Capacidade instalada reconstruida por ano. Grao: ano x subsistema x fonte. "
              "Obtida somando as unidades que ja haviam entrado em operacao e ainda nao tinham "
              "sido desativadas em 30 de junho de cada ano.",
   "ano":           "Ano de referencia. Dominio: 2019 a 2025.",
   "id_subsistema": "Codigo do subsistema. Dominio: NE, N, S, SE, PY.",
   "nom_fonte":     "Fonte de geracao. Dominio: Hidraulica, Termica, Eolica, Fotovoltaica.",
   "capacidade_mw": "Capacidade instalada em 30 de junho do ano, em MW. Dominio: maior ou igual a zero.",
 },
}

for tabela, campos in catalogo_cap.items():
    for campo, texto in campos.items():
        if campo == "_tabela":
            spark.sql(f"COMMENT ON TABLE {tabela} IS '{texto}'")
        else:
            spark.sql(f"ALTER TABLE {tabela} ALTER COLUMN {campo} COMMENT '{texto}'")
    print(f"documentado: {tabela} ({len(campos)-1} colunas)")

documentado: workspace.bronze.capacidade_geracao_bruto (20 colunas)
documentado: workspace.silver.capacidade_geracao (10 colunas)
documentado: workspace.gold.fato_capacidade (4 colunas)


## Auditoria de cobertura do catálogo

Documentar tabela por tabela está sujeito a esquecimento — durante a construção
deste projeto, colunas passaram despercebidas em três momentos distintos, em
notebooks diferentes.

A consulta abaixo elimina a inspeção manual: varre o `information_schema`, o
catálogo de metadados do próprio Unity Catalog, e retorna qualquer coluna das três
camadas que esteja sem comentário. Resultado vazio significa cobertura total.

Ela fecha o pipeline aqui, e não no notebook 05, por uma questão de ordem: este é o
último notebook a executar, e portanto o único ponto em que as onze tabelas já
existem. Verificar a documentação por consulta, e não por leitura, é o que separa um
catálogo confiável de um catálogo que apenas parece completo.

In [0]:
%sql
SELECT table_schema, table_name, column_name
FROM workspace.information_schema.columns
WHERE table_schema IN ('bronze', 'silver', 'gold')
  AND (comment IS NULL OR trim(comment) = '')
ORDER BY table_schema, table_name, ordinal_position;

table_schema,table_name,column_name
